In [ ]:
%pip install timm==0.9.12 torch torchmetrics torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade tqdm opencv-python pillow --upgrade

In [1]:
import torch
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0)) 

GPU name: NVIDIA GeForce RTX 4060 Ti


In [2]:
# Cell 1
from pathlib import Path
import hashlib, cv2, random, time
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import timm
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast
import json
import optuna
from optuna.pruners import MedianPruner
from torch.utils.data import Subset
import gc
from sklearn.metrics import fbeta_score
from tqdm.auto import tqdm

# ⚙️ SET YOUR DATA ROOT
DATA_ROOT = Path(r"G:/My Drive/CLPD-MF-Dataset")  
assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
print("Found dataset root:", DATA_ROOT)

c:\Users\Mohamed Hazem\anaconda3\envs\dlclass\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found dataset root: G:\My Drive\CLPD-MF-Dataset


In [3]:
# Cell 2
def list_images_and_labels(root):
    """
    List all images with labels and magnifications from the new folder structure.
    Adds a 'target_class' key for 5-way multi-class training.
    """
    rows = []
    root = Path(root)
    
    # Process MF folder
    mf_dir = root / "MF"
    if mf_dir.exists():
        for patient_dir in mf_dir.iterdir():
            if not patient_dir.is_dir(): continue
            
            patient_name = patient_dir.name
            if not (patient_dir / 'x10').exists():
                print(f"Warning: No x10 folder for patient {patient_name} in MF")

            for mag in ['x10', 'x20']:
                mag_dir = patient_dir / mag
                if mag_dir.exists() and mag_dir.is_dir():
                    for img_path in mag_dir.glob('*.tif'):
                        rows.append({
                            'path': img_path,
                            'label': 'MF',
                            'patient': patient_name,
                            'mag': mag,
                            'subtype': 'MF', # Set subtype to MF for consistency
                            'target_class': 'MF' # <--- NEW: Unified class for PyTorch
                        })
    
    # Process Non-MF folder with subtypes
    nonmf_dir = root / "Non-MF"
    if nonmf_dir.exists():
        for subtype_dir in nonmf_dir.iterdir():
            if not subtype_dir.is_dir(): continue
            
            subtype = subtype_dir.name  # B cell Lymphoma, PLEVA-PLC, etc.
            
            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir(): continue
                
                patient_name = patient_dir.name
                for mag in ['x10', 'x20']:
                    mag_dir = patient_dir / mag
                    if mag_dir.exists() and mag_dir.is_dir():
                        if not any(mag_dir.iterdir()):
                            print(f"Warning: No images found for patient {patient_name} in subtype {subtype} at {mag}")
                        for img_path in mag_dir.glob('*.tif'):
                            rows.append({
                                'path': img_path,
                                'label': 'Non-MF',
                                'patient': patient_name,
                                'mag': mag,
                                'subtype': subtype,
                                'target_class': subtype # <--- NEW: Maps to specific disease
                            })
            
    return rows

# Load all images
print("\n" + "="*80)
print("LOADING DATASET (5-CLASS SETUP)")
print("="*80 + "\n")

all_images = list_images_and_labels(DATA_ROOT)

print(f"Total images found: {len(all_images)}")

print(f"\nLabel distribution (Binary):")
label_counts = Counter([r['label'] for r in all_images])
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count} images")

print(f"\nTarget Class distribution (Multi-Class):")
target_counts = Counter([r['target_class'] for r in all_images])
for target, count in sorted(target_counts.items()):
    target_patients = len(set(r['patient'] for r in all_images if r['target_class'] == target))
    print(f"  {target}: {count} images ({target_patients} patients)")



LOADING DATASET (5-CLASS SETUP)

Total images found: 6257

Label distribution (Binary):
  MF: 4296 images
  Non-MF: 1961 images

Target Class distribution (Multi-Class):
  B cell Lymphoma: 358 images (18 patients)
  MF: 4296 images (310 patients)
  PLEVA-PLC: 1268 images (108 patients)
  T-cell dyscrasia: 180 images (15 patients)
  pseudolymphoma: 155 images (11 patients)


In [4]:
# Cell 3
PATCH_CACHE = Path('./patch_cache')
PATCH_CACHE.mkdir(exist_ok=True)

def extract_and_cache_patches(
    img_path,
    patch_size=512,
    stride=256,
    min_foreground_ratio=0.285,
    max_patches_per_image=150,
):
    """
    Extract tissue patches from a WSI.  Results are cached under
    PATCH_CACHE / <sha1> / ps<patch_size> / so that different
    patch sizes share the same top-level folder but are kept separate.
    """
    key       = hashlib.sha1(str(img_path).encode()).hexdigest()
    cache_dir = PATCH_CACHE / key / f"ps{patch_size}"

    if cache_dir.exists() and any(cache_dir.iterdir()):
        return sorted(str(p) for p in cache_dir.glob('*.jpg'))

    cache_dir.mkdir(parents=True, exist_ok=True)
    patches = []

    # CRITICAL FIX: Use 'with' so Windows forcefully closes the file handle immediately
    with Image.open(img_path) as img:
        W, H  = img.size

        for y in range(0, H - patch_size + 1, stride):
            for x in range(0, W - patch_size + 1, stride):
                # CRITICAL FIX: Crop FIRST, then convert only the tiny patch to RGB
                crop = img.crop((x, y, x + patch_size, y + patch_size))
                crop_rgb = crop.convert('RGB')
                
                arr  = np.asarray(crop_rgb)
                hsv  = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
                
                if (hsv[:, :, 1] > 20).mean() < min_foreground_ratio:
                    continue
                
                fname = cache_dir / f"{x}_{y}.jpg"
                crop_rgb.save(fname, quality=90)
                patches.append(str(fname))
                
                if len(patches) >= max_patches_per_image:
                    break
            if len(patches) >= max_patches_per_image:
                break

    gc.collect()
    return patches

In [5]:
# Cell 4
class MFHistologyDataset(Dataset):
        def __init__(self, rows, mag='x20', mode='train', patching=True, patch_size=512,
                     stride=256, transforms=None, max_patches_per_image=100):
            self.rows = [r for r in rows if (mag is None or r['mag']==mag)]
            self.mode = mode
            self.patching = patching
            self.patch_size = patch_size
            self.stride = stride
            self.max_patches_per_image = max_patches_per_image
            self.transforms = transforms
            
            # --- MODIFICATION 1: Use target_class instead of label ---
            labels = sorted(list({r['target_class'] for r in self.rows}))
            self.label2idx = {lab:i for i,lab in enumerate(labels)}
            # Save the reverse mapping so you know which index is which disease later
            self.idx2label = {i:lab for lab,i in self.label2idx.items()} 

            self.items = []
            for i, r in enumerate(self.rows):
                if self.patching:
                    patches = extract_and_cache_patches(r['path'], patch_size=self.patch_size,
                                                        stride=self.stride, max_patches_per_image=self.max_patches_per_image)
                    for p in patches:
                        # --- MODIFICATION 2: Save the target_class index ---
                        self.items.append({'img': p, 'label': self.label2idx[r['target_class']], 'source': str(r['path'])})
                    
                    # --- CRITICAL RAM FIX (Brought over from our earlier session) ---
                    del patches
                    gc.collect()
                    # -----------------------------------------------------------------
                else:
                    self.items.append({'img': str(r['path']), 'label': self.label2idx[r['target_class']], 'source': str(r['path'])})
                    
            if len(self.items)==0:
                print("Warning: dataset empty for magnification", mag)

        def __len__(self): return len(self.items)

        def __getitem__(self, idx):
            it = self.items[idx]
            img = Image.open(it['img']).convert('RGB')
            if self.transforms: img = self.transforms(img)
            return img, it['label'], it['source']


train_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), 
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [6]:
# Cell 5
def patient_split_stratified(rows, mag='x20', val_frac=0.15, seed=40):

    # group patients by class
    cls_map = {}
    for r in rows:
        if mag is not None and r['mag'] != mag: continue
        cls_map.setdefault(r['target_class'], {}).setdefault(r['patient'], []).append(r)

    # Now stratify
    train_rows, val_rows = [], []
    rng = random.Random(seed)

    for cls, patients_dict in cls_map.items():
        patients = list(patients_dict.keys())
        rng.shuffle(patients)

        n = len(patients)
        n_val = max(1, int(n * val_frac))
        val_p = set(patients[:n_val]) 

        for p, rlist in patients_dict.items():
            if p in val_p:  
                val_rows += rlist
            else: 
                train_rows += rlist
    return train_rows, val_rows

# Create splits
train_val_rows, test_rows = patient_split_stratified(all_images, mag=None, val_frac=0.15, seed=40)
val_frac_adjusted = 0.15 / (1 - 0.15)
train_rows, val_rows = patient_split_stratified(train_val_rows, mag=None, val_frac=val_frac_adjusted, seed=40)

print(f"Total patients: {len(set(r['patient'] for r in all_images))}")
print(f"Training patients: {len(set(r['patient'] for r in train_rows))}")
print(f"Validation patients: {len(set(r['patient'] for r in val_rows))}")
print(f"Test patients: {len(set(r['patient'] for r in test_rows))}")

# --- Create Datasets for x20 ---
train_rows_20 = [r for r in train_rows if r['mag'] == 'x20']
val_rows_20 = [r for r in val_rows if r['mag'] == 'x20']
test_rows_20 = [r for r in test_rows if r['mag'] == 'x20'] 

# Explicitly set patch_size=1024 and stride=512 for 20x
train_ds_20 = MFHistologyDataset(train_rows_20, mag='x20', mode='train', patching=True, patch_size=1024, stride=512, transforms=train_tf)
val_ds_20   = MFHistologyDataset(val_rows_20, mag='x20', mode='val', patching=True, patch_size=1024, stride=512, transforms=val_tf)
test_ds_20  = MFHistologyDataset(test_rows_20, mag='x20', mode='test', patching=True, patch_size=1024, stride=512, transforms=val_tf) 
print(f"\nx20 Datasets (patches): Train={len(train_ds_20)}, Val={len(val_ds_20)}, Test={len(test_ds_20)}")

# --- Create Datasets for x10 ---
train_rows_10 = [r for r in train_rows if r['mag'] == 'x10']
val_rows_10 = [r for r in val_rows if r['mag'] == 'x10']
test_rows_10 = [r for r in test_rows if r['mag'] == 'x10'] 

# Explicitly set patch_size=512 and stride=256 for 10x
train_ds_10 = MFHistologyDataset(train_rows_10, mag='x10', mode='train', patching=True, patch_size=512, stride=256, transforms=train_tf)
val_ds_10   = MFHistologyDataset(val_rows_10, mag='x10', mode='val', patching=True, patch_size=512, stride=256, transforms=val_tf)
test_ds_10  = MFHistologyDataset(test_rows_10, mag='x10', mode='test', patching=True, patch_size=512, stride=256, transforms=val_tf)
print(f"x10 Datasets (patches): Train={len(train_ds_10)}, Val={len(val_ds_10)}, Test={len(test_ds_10)}")

def create_model(model_name='tf_efficientnet_b3', pretrained=True, num_classes=5, dropout=0.4):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    return model

Total patients: 462
Training patients: 328
Validation patients: 67
Test patients: 67

x20 Datasets (patches): Train=23062, Val=4728, Test=4407
x10 Datasets (patches): Train=73824, Val=14804, Test=13190


In [ ]:
# Cell 6 - Optuna Hyperparameter Tuning

# Configuration
OPTUNA_N_EPOCHS = 1
OPTUNA_N_TRIALS = 30
OPTUNA_DB = "optuna_mf.db"
SUBSAMPLE_RATIO = 0.15

def create_subsampled_dataset(dataset, ratio=0.15, seed=42):
    """Create subsampled dataset using Subset"""
    n_samples = len(dataset)
    n_subset = int(n_samples * ratio)
    indices = torch.randperm(n_samples, generator=torch.Generator().manual_seed(seed))[:n_subset].tolist()
    return Subset(dataset, indices)

def objective_fn(trial, train_subset, val_subset, mag, device):
    """Optuna objective function"""
    # Hyperparameters
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    model_arch = trial.suggest_categorical('model_architecture', ['tf_efficientnet_b3', 'resnet50'])
    
    try:
        # --- FIX 1: num_classes=5 for our new differential diagnosis setup ---
        model = create_model(model_name=model_arch, pretrained=True, num_classes=5, dropout=dropout).to(device)
        
        # DataLoaders
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
        
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        # Compute class weights from subset
        counts = {}
        for idx in train_subset.indices:
            label = train_subset.dataset.items[idx]['label']
            counts[label] = counts.get(label, 0) + 1
        total = sum(counts.values())
        
        # --- FIX 2: range(5) to dynamically weight all 5 classes ---
        weights = torch.tensor([total/counts.get(i, 1) for i in range(5)], dtype=torch.float).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        
        scaler = torch.amp.GradScaler('cuda')
        
        # Training loop
        for epoch in range(OPTUNA_N_EPOCHS):
            # Train
            model.train()
            for imgs, labels, _ in train_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with autocast('cuda'):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            # Validate
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for imgs, labels, _ in val_loader:
                    imgs = imgs.to(device)
                    labels = labels.to(device)
                    outputs = model(imgs)
                    preds = outputs.argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total += imgs.size(0)
            
            val_acc = correct / total if total > 0 else 0.0
            
            # Report for pruning
            trial.report(val_acc, epoch)
            
            # Check pruning
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return val_acc
        
    finally:
        # Clean up GPU memory
        del model
        if 'optimizer' in locals(): del optimizer
        if 'criterion' in locals(): del criterion
        if 'scaler' in locals(): del scaler
        torch.cuda.empty_cache()
        gc.collect()

# Create subsampled datasets
print("\nCreating subsampled datasets for Optuna...")
train_subset_10 = create_subsampled_dataset(train_ds_10, ratio=SUBSAMPLE_RATIO, seed=42)
val_subset_10 = create_subsampled_dataset(val_ds_10, ratio=SUBSAMPLE_RATIO, seed=42)
train_subset_20 = create_subsampled_dataset(train_ds_20, ratio=SUBSAMPLE_RATIO, seed=42)
val_subset_20 = create_subsampled_dataset(val_ds_20, ratio=SUBSAMPLE_RATIO, seed=42)

print(f"x10 subsampled: Train={len(train_subset_10)}, Val={len(val_subset_10)}")
print(f"x20 subsampled: Train={len(train_subset_20)}, Val={len(val_subset_20)}")

# Pruner
pruner = MedianPruner(n_startup_trials=1, n_warmup_steps=0)

# Tune x10
print("\n" + "="*80)
print("TUNING x10 MODEL")
print("="*80)

storage_x10 = f"sqlite:///{OPTUNA_DB}"
study_x10 = optuna.create_study(
    study_name="mf_x10_tuning",
    storage=storage_x10,
    direction="maximize",
    pruner=pruner,
    load_if_exists=True
)

study_x10.optimize(
    lambda trial: objective_fn(trial, train_subset_10, val_subset_10, 'x10', 'cuda'),
    n_trials=OPTUNA_N_TRIALS,
    show_progress_bar=True
)

best_params_x10 = study_x10.best_params
best_value_x10 = study_x10.best_value

print(f"\nBest x10 validation accuracy: {best_value_x10:.4f}")
print(f"Best x10 parameters: {best_params_x10}")

with open('best_params_x10.json', 'w') as f:
    json.dump({'best_params': best_params_x10, 'best_value': best_value_x10, 'n_trials': len(study_x10.trials)}, f, indent=2)

torch.cuda.empty_cache()
gc.collect()

# Tune x20
print("\n" + "="*80)
print("TUNING x20 MODEL")
print("="*80)

storage_x20 = f"sqlite:///{OPTUNA_DB}"
study_x20 = optuna.create_study(
    study_name="mf_x20_tuning",
    storage=storage_x20,
    direction="maximize",
    pruner=pruner,
    load_if_exists=True
)

study_x20.optimize(
    lambda trial: objective_fn(trial, train_subset_20, val_subset_20, 'x20', 'cuda'),
    n_trials=OPTUNA_N_TRIALS,
    show_progress_bar=True
)

best_params_x20 = study_x20.best_params
best_value_x20 = study_x20.best_value

print(f"\nBest x20 validation accuracy: {best_value_x20:.4f}")
print(f"Best x20 parameters: {best_params_x20}")

with open('best_params_x20.json', 'w') as f:
    json.dump({'best_params': best_params_x20, 'best_value': best_value_x20, 'n_trials': len(study_x20.trials)}, f, indent=2)

torch.cuda.empty_cache()
gc.collect()

print("\n" + "="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)

In [7]:
# Cell 7 - Final Training with Golden Defaults

# Save Path
save_path = Path("C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/Mycosis-Fungoides-Classifier/Trained Models")
save_path.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda")

print("\n" + "="*80)
print("LOADING GOLDEN HYPERPARAMETERS (Bypassing Optuna)")
print("="*80)

# --- HARDCODED BEST CONFIGURATIONS ---
ARCH = 'tf_efficientnet_b3'
LR = 2e-4
WEIGHT_DECAY = 1e-2
DROPOUT = 0.4
EPOCHS = 10

# x10 Specifics
BS_10 = 16
ACCUM_10 = 1

# x20 Specifics
BS_20 = 2
ACCUM_20 = 8

def compute_f2(preds, labels):
    if len(preds) == 0: return 0.0
    return fbeta_score(labels.numpy(), preds.numpy(), beta=2, average='macro', zero_division=0)

# --- Gradient Accumulation Training Loop (WITH PROGRESS BAR) ---
def train_one_epoch_accum(model, loader, optimizer, criterion, device, scaler, accum_steps):
    model.train()
    running_loss, total, correct = 0.0, 0, 0
    optimizer.zero_grad() 
    
    # Wrap the loader in tqdm for a dynamic progress bar
    pbar = tqdm(loader, desc="  Training", leave=False)
    
    for batch_idx, (imgs, labels, _src) in enumerate(pbar):
        imgs, labels = imgs.to(device), labels.to(device)
        
        with autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss = loss / accum_steps # Normalize for accumulation
            
        scaler.scale(loss).backward()
        
        if ((batch_idx + 1) % accum_steps == 0) or (batch_idx + 1 == len(loader)):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        running_loss += (loss.item() * accum_steps) * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        
        # Update the progress bar with live metrics!
        pbar.set_postfix({'Loss': f"{running_loss/total:.4f}", 'Acc': f"{correct/total:.4f}"})
        
    return running_loss/total, correct/total

# --- Validation Loop (WITH PROGRESS BAR) ---
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, total, correct = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc="  Validating", leave=False)
    
    with torch.no_grad():
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            
            pbar.set_postfix({'Loss': f"{running_loss/total:.4f}"})
    
    avg_loss = running_loss / total if total > 0 else 0
    acc = correct / total if total > 0 else 0
    all_preds = torch.cat(all_preds) if all_preds else torch.tensor([])
    all_labels = torch.cat(all_labels) if all_labels else torch.tensor([])
    f2 = compute_f2(all_preds, all_labels)
    
    return avg_loss, acc, f2

def get_class_weights(train_ds):
    counts = {}
    for it in train_ds.items:
        counts[it['label']] = counts.get(it['label'], 0) + 1
    total = sum(counts.values())
    weights = [total/counts.get(i, 1) for i in range(len(counts))]
    return torch.tensor(weights, dtype=torch.float).to(device)

# Dataloaders
train_loader_10 = DataLoader(train_ds_10, batch_size=BS_10, shuffle=True, num_workers=0, pin_memory=True)
val_loader_10   = DataLoader(val_ds_10, batch_size=BS_10, shuffle=False, num_workers=0, pin_memory=True)

train_loader_20 = DataLoader(train_ds_20, batch_size=BS_20, shuffle=True, num_workers=0, pin_memory=True)
val_loader_20   = DataLoader(val_ds_20, batch_size=BS_20, shuffle=False, num_workers=0, pin_memory=True)

# Create Models (5-class)
model_10 = create_model(model_name=ARCH, pretrained=True, num_classes=5, dropout=DROPOUT).to(device)
model_20 = create_model(model_name=ARCH, pretrained=True, num_classes=5, dropout=DROPOUT).to(device)

# Compute Weights & Loss
weights_10 = get_class_weights(train_ds_10)
weights_20 = get_class_weights(train_ds_20)
criterion_10 = nn.CrossEntropyLoss(weight=weights_10)
criterion_20 = nn.CrossEntropyLoss(weight=weights_20)

# Optimizers & Scalers
optimizer_10 = optim.AdamW(model_10.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
optimizer_20 = optim.AdamW(model_20.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler_10 = torch.amp.GradScaler('cuda')
scaler_20 = torch.amp.GradScaler('cuda')

best_val_f2_10 = 0.0
best_val_f2_20 = 0.0

print("\n" + "="*80)
print(f"STARTING FINAL TRAINING: {ARCH} (5-Class)")
print(f"Epochs: {EPOCHS} | Dropout: {DROPOUT} | LR: {LR}")
print("="*80)

# Training loop
for epoch in range(EPOCHS):
    t0 = time.time()

    # Train x10 model
    train_loss_10, train_acc_10 = train_one_epoch_accum(model_10, train_loader_10, optimizer_10, criterion_10, device, scaler_10, ACCUM_10)
    val_loss_10, val_acc_10, val_f2_10 = validate(model_10, val_loader_10, criterion_10, device)

    # Train x20 model
    train_loss_20, train_acc_20 = train_one_epoch_accum(model_20, train_loader_20, optimizer_20, criterion_20, device, scaler_20, ACCUM_20)
    val_loss_20, val_acc_20, val_f2_20 = validate(model_20, val_loader_20, criterion_20, device)

    t1 = time.time()
    print(
    f"Epoch {epoch+1}/{EPOCHS} | "
    f"x10 Val F2 {val_f2_10:.4f} (Acc {val_acc_10:.4f}) | "
    f"x20 Val F2 {val_f2_20:.4f} (Acc {val_acc_20:.4f}) | "
    f"time {(t1-t0):.1f}s")

    # Save best models
    if val_f2_10 > best_val_f2_10:
        best_val_f2_10 = val_f2_10
        torch.save(model_10.state_dict(), save_path / f'model_{ARCH}_x10_5class.pth')
        print(f"  ✓ Saved BEST x10 model")
    
    if val_f2_20 > best_val_f2_20:
        best_val_f2_20 = val_f2_20
        torch.save(model_20.state_dict(), save_path / f'model_{ARCH}_x20_5class.pth')
        print(f"  ✓ Saved BEST x20 model")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print(f"Best x10 validation F2: {best_val_f2_10:.4f}")
print(f"Best x20 validation F2: {best_val_f2_20:.4f}")


LOADING GOLDEN HYPERPARAMETERS (Bypassing Optuna)

STARTING FINAL TRAINING: tf_efficientnet_b3 (5-Class)
Epochs: 10 | Dropout: 0.4 | LR: 0.0002


Epoch 1/10 | x10 Val F2 0.3843 (Acc 0.5894) | x20 Val F2 0.2687 (Acc 0.3479) | time 4643.8s
  ✓ Saved BEST x10 model
  ✓ Saved BEST x20 model


Epoch 2/10 | x10 Val F2 0.3876 (Acc 0.6347) | x20 Val F2 0.3403 (Acc 0.5459) | time 3857.4s
  ✓ Saved BEST x10 model
  ✓ Saved BEST x20 model


Epoch 3/10 | x10 Val F2 0.4170 (Acc 0.5852) | x20 Val F2 0.3246 (Acc 0.5102) | time 3825.6s
  ✓ Saved BEST x10 model


Epoch 4/10 | x10 Val F2 0.3998 (Acc 0.6304) | x20 Val F2 0.3681 (Acc 0.5501) | time 3839.4s
  ✓ Saved BEST x20 model


...

Epoch 6/10 | x10 Val F2 0.4084 (Acc 0.6394) | x20 Val F2 0.3350 (Acc 0.5920) | time 3807.0s


Epoch 7/10 | x10 Val F2 0.4096 (Acc 0.6447) | x20 Val F2 0.3501 (Acc 0.5459) | time 3809.4s


Epoch 8/10 | x10 Val F2 0.4129 (Acc 0.6745) | x20 Val F2 0.3463 (Acc 0.5778) | time 3799.4s


Epoch 9/10 | x10 Val F2 0.3998 (Acc 0.6548) | x20 Val F2 0.3492 (Acc 0.6013) | time 3806.6s


Epoch 10/10 | x10 Val F2 0.4110 (Acc 0.6652) | x20 Val F2 0.3572 (Acc 0.6246) | time 3812.3s

TRAINING COMPLETE
Best x10 validation F2: 0.4207
Best x20 validation F2: 0.3681
